# Preprocessing and Data Split - Local Version
This notebook creates 3 preprocessing versions:
1. **Raw**: Original cropped images (reference from data/Cropped)
2. **Unified**: Standard preprocessing pipeline for all categories
3. **Customized**: Category-specific preprocessing pipelines

In [1]:
import cv2
import os
import numpy as np
from glob import glob
from tqdm import tqdm
import json

# 1. SETUP LOCAL PATHS (RELATIVE TO WORKSPACE)
RAW_PATH = os.path.join("..", "data", "Cropped")
OUTPUT_UNIFIED = os.path.join("..", "data", "processed_unified")
OUTPUT_CUSTOM = os.path.join("..", "data", "processed_customized")
CATEGORIES = ['Can', 'Paper', 'Plastic Bag', 'Plastic Bottle']

# Create local directories
for root in [OUTPUT_UNIFIED, OUTPUT_CUSTOM]:
    for cat in CATEGORIES:
        os.makedirs(os.path.join(root, cat), exist_ok=True)
        
print(f"Raw data path: {RAW_PATH}")
print(f"Unified output: {OUTPUT_UNIFIED}")
print(f"Custom output: {OUTPUT_CUSTOM}")

Raw data path: ..\data\Cropped
Unified output: ..\data\processed_unified
Custom output: ..\data\processed_customized


In [2]:
# 2. UNIFIED PREPROCESSING STACK
def apply_unified_stack(img):
    """Deterministic Preprocessing: CLAHE + Gaussian Blur + Canny"""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(gray)
    blurred = cv2.GaussianBlur(clahe, (5, 5), 0)
    edges = cv2.Canny(blurred, 50, 150)
    return cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

In [ ]:
# 3. CUSTOMIZED PREPROCESSING STACKS (Category-specific)
def apply_can_stack(img):
    """Can-specific: Gaussian Blur + Sharpening + Canny"""
    blurred = cv2.GaussianBlur(img, (5, 5), 0)
    kernel_sharpening = np.array([[-1,-1,-1], [-1,9,-1], [-1,-1,-1]])
    sharpened = cv2.filter2D(blurred, -1, kernel_sharpening)
    gray = cv2.cvtColor(sharpened, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 100, 200)
    return cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

def apply_paper_stack(img):
    """Paper-specific: Simple thresholding"""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
    return cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)

def apply_plastic_bag_stack(img):
    """Plastic Bag-specific: CLAHE + Median Filtering + Flip"""
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    enhanced_l = clahe.apply(l)
    img_enhanced = cv2.cvtColor(cv2.merge((enhanced_l, a, b)), cv2.COLOR_LAB2BGR)
    filtered = cv2.medianBlur(img_enhanced, 5)
    transformed = cv2.flip(filtered, 1)  # Horizontal flip
    return transformed

def apply_plastic_bottle_stack(img):
    """Plastic Bottle-specific: Contrast + Bilateral Filter + Threshold"""
    contrast = cv2.convertScaleAbs(img, alpha=1.3, beta=10)
    bilateral = cv2.bilateralFilter(contrast, 9, 75, 75)
    gray = cv2.cvtColor(bilateral, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
    return cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)

In [4]:
# 4. BATCH PROCESSING EXECUTION
def run_processing(mode='unified'):
    target_path = OUTPUT_UNIFIED if mode == 'unified' else OUTPUT_CUSTOM
    print(f"🚀 Starting {mode.upper()} preprocessing...")
    
    for cat in CATEGORIES:
        files = glob(os.path.join(RAW_PATH, cat, "*.[jJ][pP][gG]"))
        print(f"Found {len(files)} files for {cat}")
        
        for f in tqdm(files, desc=f"Processing {cat}"):
            img = cv2.imread(f)
            if img is None: 
                print(f"Warning: Could not read {f}")
                continue
            
            # Standardize size
            img = cv2.resize(img, (512, 512))
            
            if mode == 'unified':
                res = apply_unified_stack(img)
            else:
                # Apply specific stack based on category
                if cat == 'Can': 
                    res = apply_can_stack(img)
                elif cat == 'Paper': 
                    res = apply_paper_stack(img)
                elif cat == 'Plastic Bag': 
                    res = apply_plastic_bag_stack(img)
                elif cat == 'Plastic Bottle': 
                    res = apply_plastic_bottle_stack(img)
                else: 
                    res = img  # Fallback
                
            output_file = os.path.join(target_path, cat, os.path.basename(f))
            cv2.imwrite(output_file, res)
    
    print(f"✅ {mode.upper()} preprocessing complete!")

In [5]:
# 5. Execute Both Preprocessing Modes
run_processing(mode='unified')
run_processing(mode='customized')

🚀 Starting UNIFIED preprocessing...
Found 1265 files for Can


Processing Can: 100%|██████████| 1265/1265 [00:49<00:00, 25.50it/s]


Found 3035 files for Paper


Processing Paper: 100%|██████████| 3035/3035 [01:48<00:00, 28.02it/s]


Found 949 files for Plastic Bag


...

Found 949 files for Plastic Bag


Processing Plastic Bag: 100%|██████████| 949/949 [00:15<00:00, 61.92it/s]


Found 3781 files for Plastic Bottle


Processing Plastic Bottle: 100%|██████████| 3781/3781 [02:24<00:00, 26.22it/s]

✅ CUSTOMIZED preprocessing complete!


In [6]:
# 6. Verify Output
print("\n📊 PREPROCESSING SUMMARY:")
print("=" * 40)

for mode, path in [("RAW", RAW_PATH), ("UNIFIED", OUTPUT_UNIFIED), ("CUSTOMIZED", OUTPUT_CUSTOM)]:
    print(f"\n{mode} Data:")
    total = 0
    for cat in CATEGORIES:
        cat_path = os.path.join(path, cat)
        if os.path.exists(cat_path):
            count = len(glob(os.path.join(cat_path, "*.[jJ][pP][gG]")))
            print(f"  {cat}: {count} images")
            total += count
        else:
            print(f"  {cat}: Directory not found")
    print(f"  TOTAL: {total} images")


📊 PREPROCESSING SUMMARY:

RAW Data:
  Can: 1265 images
  Paper: 3035 images
  Plastic Bag: 949 images
  Plastic Bottle: 3781 images
  TOTAL: 9030 images

UNIFIED Data:
  Can: 1265 images
  Paper: 3035 images
  Plastic Bag: 949 images
  Plastic Bottle: 3781 images
  TOTAL: 9030 images

CUSTOMIZED Data:
  Can: 1265 images
  Paper: 3035 images
  Plastic Bag: 949 images
  Plastic Bottle: 3781 images
  TOTAL: 9030 images
